# Аналитика бизнес-сообществ ВКонтакте

Анализирую продавцов товаров: изучаю состав сообществ, контент, каталог, подключённые инструменты, рекламные расходы и заказы через чекаут. Сравниваю сегменты и проверяю гипотезы, чтобы выявить особенности продавцов и возможные препятствия продаже.

Источник — `data/businesses.csv`. Период — 1–30 сентября 2026 года. Данные синтетические, подготовлены для VK Case Champ и не отражают реальные показатели VK. Исходный файл сохраняю без изменений; подготовку выборки и расчёты выполняю в ноутбуке.


## Часть 1. Подготовка к анализу

Импорты и путь к исходному CSV. Рабочая папка — корень репозитория.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
# Ноутбук запускается из корня репозитория.
ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "businesses.csv"

## Часть 2. Структура и смысл данных

Загружаем исходный CSV стандартным способом: pandas автоматически определяет типы и распознаёт пропуски. Очистку и фильтрацию не выполняем.


In [3]:
df = pd.read_csv(DATA_PATH, encoding="utf-8")

print(f"Строк: {len(df):,}; столбцов: {df.shape[1]}")

Строк: 9,754; столбцов: 28


### 2.1. Исходные данные: первые 10 строк

`df` содержит все строки и столбцы исходного CSV. Показываем первые 10 строк. Типы и пропуски распознаны pandas автоматически; очистки, фильтрации и заполнения значений не было. Индекс слева добавлен pandas и не является полем источника. CSV не изменяется.


In [4]:
display(df.head(10))

,ID бизнеса,Тематика,Тип бизнеса (товары / услуги),Число подписчиков,Посты за 30 дней,Клипы за 30 дней,Посты с товаром за 30 дней,Клипы с товаром за 30 дней,Просмотры обычного контента,Лайки обычного контента,Репосты обычного контента,Просмотры товарного контента,Лайки товарного контента,Репосты товарного контента,Открытия карточки товара из контента,Товары загружены,Число активных товаров в каталоге,Подключён чекаут,Подключена доставка,Подключена оплата,Автор оригинального контента (оценка модели),Участие в реферальной программе,Заказы через чекаут,"GMV через чекаут, ₽","Средний чек, ₽","Траты на рекламу всего, ₽",Число активных услуг,"Медианная цена товара, ₽"
0,9308,Мода и стиль,без каталога,2800,66,1,22,1,1000000,51694,268,3800000,103250,1287,8372,0,0,0,0,0,1.0,1,0,0,NaN,0,0,NaN
1,3078,Спорт,без каталога,98000,7,46,0,7,550000,16429,10479,2300000,49299,28344,5818,0,0,1,0,0,1.0,1,0,0,NaN,0,0,NaN
2,5330,Образование,товарное,12000,380,0,26,0,380000,2530,636,1300000,1999,1065,6749,700,693,1,1,1,1.0,1,83,17150,208.0,0,0,169.0
3,5072,Развлечения и хобби,товарное,5200,12,0,20,0,260000,319,37,1200000,1380,1266,550,4986,4887,1,1,1,0.0,0,2,6250,2648.0,0,0,1325.0
4,6398,Личный блог,без каталога,440,35,20,7,12,530000,1915,455,1100000,2584,1228,10859,0,0,0,0,0,1.0,1,0,0,NaN,0,0,NaN
5,4267,Путешествия и отдых,товарное и услужное,22000,35,18,15,6,730000,2062,223,900000,1168,401,778,130,106,0,0,0,0.0,0,0,0,NaN,110000,31,4240.0
6,2,Путешествия и отдых,товарное,1400,3,26,0,23,3600,338,1,540000,1754,566,1268,268,250,1,0,0,1.0,1,0,0,NaN,0,0,466.0
7,134,Образование,товарное,5800,0,2,0,1,5400,114,8,490000,1889,2036,796,28,28,1,1,0,0.0,0,0,0,NaN,0,0,20140.0
8,4977,Творчество и хендмейд,товарное и услужное,8200,11,12,2,10,36000,38,10,460000,205,19,232,44,44,1,1,0,1.0,0,7,5350,756.0,0,4,530.0
9,2932,Личный блог,без каталога,5400,19,7,2,2,460000,2770,102,440000,925,475,3711,0,0,0,0,0,1.0,1,0,0,NaN,0,0,NaN


### 2.2. Столбцы и типы данных

CSV не хранит схему типов. Ниже — типы, автоматически определённые pandas при загрузке. ID остаётся идентификатором, даже если распознан как число. Флаг может иметь тип `float64` из-за пропусков. Тип сам по себе не подтверждает бизнес-смысл или корректность значений.


In [5]:
schema = pd.DataFrame({
    "№": range(1, len(df.columns) + 1),
    "Столбец": df.columns,
    "Тип pandas": df.dtypes.astype(str).values,
})
display(schema)

,№,Столбец,Тип pandas
0,1,ID бизнеса,int64
1,2,Тематика,str
2,3,Тип бизнеса (товары / услуги),str
3,4,Число подписчиков,int64
4,5,Посты за 30 дней,int64
5,6,Клипы за 30 дней,int64
6,7,Посты с товаром за 30 дней,int64
7,8,Клипы с товаром за 30 дней,int64
8,9,Просмотры обычного контента,int64
9,10,Лайки обычного контента,int64


**Решение по типам данных.** Текущие типы оставляем без изменений. С учётом дальнейшего построения модели принято решение не переводить признаки подключения чекаута, доставки, оплаты и участия в реферальной программе из `int64` в `boolean`: сохраняем кодировку `0/1`. Оценку оригинальности оставляем в `float64` с пропусками, тематику и тип бизнеса — текстовыми. Перед моделированием отдельно подготовим признаки: обработаем пропуски, закодируем текстовые категории и при необходимости масштабируем числовые значения, сохранив исходный `df`.


### 2.3. Словарь полей

**Период данных: с 1 по 30 сентября 2026 года включительно.**

#### Бизнес и аудитория

| Поле | Смысл | Единица |
|---|---|---|
| ID бизнеса | Уникальный идентификатор бизнеса; ключ строки в этом наборе данных. | ID |
| Тематика | Тематическая категория сообщества. | Категория |
| Тип бизнеса (товары / услуги) | Категория товарного/услужного профиля и наличия каталога. | Категория |
| Число подписчиков | Число подписчиков сообщества. | Подписчики |


#### Контент и интерес к товарам

| Поле | Смысл | Единица |
|---|---|---|
| Посты за 30 дней | Количество постов за 30 дней. | Публикации |
| Клипы за 30 дней | Количество клипов за 30 дней. | Публикации |
| Посты с товаром за 30 дней | Количество постов с товаром. | Публикации |
| Клипы с товаром за 30 дней | Количество клипов с товаром. | Публикации |
| Просмотры обычного контента | Просмотры обычного контента. | Просмотры (события) |
| Лайки обычного контента | Лайки обычного контента. | Лайки (события) |
| Репосты обычного контента | Репосты обычного контента. | Репосты (события) |
| Просмотры товарного контента | Просмотры товарного контента. | Просмотры (события) |
| Лайки товарного контента | Лайки товарного контента. | Лайки (события) |
| Репосты товарного контента | Репосты товарного контента. | Репосты (события) |
| Открытия карточки товара из контента | Открытия карточек товара из контента. | Открытия (события) |


#### Каталог и инструменты

| Поле | Смысл | Единица |
|---|---|---|
| Товары загружены | Количество загруженных товаров. | Товары |
| Число активных товаров в каталоге | Количество активных товаров в каталоге. | Товары |
| Подключён чекаут | Признак подключения оформления заказа внутри платформы. | 0/1 |
| Подключена доставка | Признак подключения доставки. | 0/1 |
| Подключена оплата | Признак подключения оплаты. | 0/1 |
| Автор оригинального контента (оценка модели) | Модельная оценка авторства оригинального контента на уровне сообщества. | 0/1 и пропуски |
| Участие в реферальной программе | Признак участия в реферальной программе. | 0/1 |


#### Заказы, реклама и ассортимент

| Поле | Смысл | Единица |
|---|---|---|
| Заказы через чекаут | Количество заказов через чекаут. | Заказы |
| GMV через чекаут, ₽ | Сумма заказов через чекаут без вычета отмен, возвратов и комиссий. | Рубли |
| Средний чек, ₽ | Средний чек; по материалам GMV / число заказов. | Рубли на заказ |
| Траты на рекламу всего, ₽ | Суммарные рекламные расходы. | Рубли |
| Число активных услуг | Количество активных услуг. | Услуги |
| Медианная цена товара, ₽ | Медианная цена товара. | Рубли на товар |

**Существенные ограничения:**

- Неясно, входят ли товарные посты и клипы в общие счётчики и какие форматы входят в просмотры. До уточнения нельзя уверенно рассчитать просмотры на публикацию и долю товарного контента.
- Просмотры и открытия не равны уникальным пользователям. Открытия из контента и все заказы через чекаут не образуют подтверждённую сквозную воронку.
- Заказы в сообщениях и на внешних сайтах не наблюдаются: ноль заказов через чекаут не означает отсутствие продаж.
- Оригинальность — оценка модели с неизвестным качеством; пропуск не равен нулю.
- **Допущение о подключении инструментов:** даты подключения не предоставлены. Считаем, что инструменты, отмеченные как подключённые (`1`), были подключены на начало периода — 1 сентября 2026 года — и оставались подключёнными весь месяц. Это аналитическое допущение, а не подтверждённая история подключений.


### 2.4. Охват выборки

Количество строк, пустых и уникальных ID, а также распределение по типам бизнеса до фильтрации.


In [7]:
id_col = "ID бизнеса"
coverage = pd.DataFrame([
    ("Строк в исходнике", len(df)),
    ("Пустых ID", int(df[id_col].isna().sum())),
    ("Уникальных непустых ID", df[id_col].nunique()),
], columns=["Показатель", "Значение"])
display(coverage)

business_types = (
    df["Тип бизнеса (товары / услуги)"]
    .value_counts(dropna=False)
    .rename_axis("Тип бизнеса")
    .reset_index(name="Количество сообществ")
)
business_types["Доля, %"] = (
    business_types["Количество сообществ"] / len(df) * 100
).round(2)
display(business_types)


,Показатель,Значение
0,Строк в исходнике,9754
1,Пустых ID,0
2,Уникальных непустых ID,9754


,Тип бизнеса,Количество сообществ,"Доля, %"
0,без каталога,3168,32.48
1,товарное,3150,32.29
2,услужное,2428,24.89
3,товарное и услужное,1008,10.33


### Решение по составу аналитической выборки

Исходный `df` сохраняем полностью. При подготовке отдельной аналитической выборки применяем следующие правила:

| Тип бизнеса | Решение |
|---|---|
| `товарное` | Оставляем. |
| `товарное и услужное` | Оставляем как отдельную группу. |
| `услужное` | Исключаем из основной аналитики: фокус на продаже товаров. |
| `без каталога` | Проверяем товарные поля. Исключаем, если одновременно `Товары загружены = 0` и `Число активных товаров в каталоге = 0`. Если есть ненулевое значение, сохраняем для отдельного разбора. |

Пропуск не приравниваем к нулю. Противоречивые записи проверяем на этапе качества. Отсутствие товаров в этих полях используем как правило отбора по данным, а не как доказательство отсутствия продаж вообще.

Для сравнений формируем три представления одной подготовленной выборки:

1. **Товарные** — только тип `товарное`.
2. **Товарно-услужные** — только тип `товарное и услужное`.
3. **Объединённая выборка** — обе группы вместе с сохранённым признаком типа бизнеса.

Так можно сравнивать группы отдельно и анализировать их совместно, учитывая тип бизнеса. Оставшиеся записи `без каталога` рассматриваем отдельно, не присваивая им товарный тип автоматически. По участию в реферальной программе строки не исключаем. Признак сохраняем для возможного сравнения групп; он сам по себе не определяет, является ли сообщество бизнесом или автором.

На этом этапе зафиксированы только правила; фильтрация ещё не выполнена.


### 2.5. Доступные показатели и особенности интерпретации


В данных представлены показатели контента и интереса к товарам, состояние торговых инструментов, ассортимент и цены, рекламные расходы, заказы и GMV через чекаут.

**Важный момент про воронку:** открытия карточек учтены только из контента, а заказы через чекаут могут поступать из разных источников. Связи между конкретным открытием и последующим заказом в таблице нет. Поэтому по этим агрегатам нельзя определить сквозную конверсию «открытие карточки из контента → заказ».

Продажи в сообщениях и на внешних сайтах в CSV не представлены. Ноль заказов через чекаут не означает отсутствие продаж вообще. GMV — сумма заказов, а не прибыль бизнеса или выручка VK.

**Уникальные пользователи:** отдельной метрики в CSV нет. Просмотры и открытия карточек — события, которые один человек мог совершать несколько раз. Поэтому по ним нельзя определить число уникальных пользователей или уникальный охват. Число подписчиков также не заменяет эти показатели.


### 2.6. Что нужно уточнить до дальнейших расчётов


| Тема | Вопрос |
| --- | --- |
| Счётчики публикаций | Входят ли посты и клипы с товаром в общее число постов и клипов за 30 дней? |
| Допущение о подключённых инструментах | Допустимо ли считать, что чекаут, доставка и оплата, отмеченные как подключённые, были подключены с 1 сентября 2026 года и работали весь месяц? |
| Трактовка заказов через чекаут | Как трактовать столбец «Заказы через чекаут»: учитываются только заказы после перехода из контента или также заказы, пришедшие из витрины и других источников? |
| Период публикации контента | Сентябрьские просмотры и реакции относятся только к публикациям сентября или также к публикациям предыдущих месяцев? |
| Медианная цена товара | Как рассчитывается медианная цена товара: по всем загруженным или только активным товарам? |

Эти уточнения важны для корректного расчёта доли товарного контента и просмотров на публикацию, сравнения сообществ по подключённым инструментам и цене товаров, а также интерпретации связи открытий карточек с заказами через чекаут.
